In [ ]:
import sys, os, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install",
    "flask", "flask-cors", "pyngrok", "openai", "requests", "-q"])
for url in [
    "https://raw.githubusercontent.com/omar-florez/agent-coliseum/bcp/agent_base.py",
    "https://raw.githubusercontent.com/omar-florez/agent-coliseum/bcp/agent_server.py",
]:
    subprocess.check_call(["wget", "-q", "-N", url])
os.environ["AZURE_API_KEY"]  = "YOUR_AZURE_API_KEY_HERE"
os.environ["AZURE_BASE_URL"] = "https://rsgd15-foundry.openai.azure.com/openai/v1/"
os.environ["NGROK_TOKEN"]    = "YOUR_NGROK_TOKEN_HERE"
print(f"Python: {sys.executable}")
print("Setup complete")

In [ ]:
# ============================================================
#  AGENT COLISEUM — BCP Branch — Colab 02: LangChain-style Agent
# ============================================================
#  Strategy: Structured memory agent.
#    - Uses plain openai client (avoids LangChain/transformers conflicts)
#    - Maintains per-match history as a plain list
#    - 6-step CoT reasoning (SITUATION/OPPONENT/GOAL/DRAFT/CRITIQUE/FINAL)
#    - Demonstrates how LangChain chain patterns map to the arena API
# ============================================================

import os, json, random
from openai import OpenAI
from agent_base import Agent, MatchContext, MatchResult, WorldContext, Position
from agent_server import serve_and_register

# ── Config ────────────────────────────────────────────────────────────────
AZURE_API_KEY  = os.environ.get("AZURE_API_KEY",  "YOUR_AZURE_API_KEY_HERE")
AZURE_BASE_URL = os.environ.get("AZURE_BASE_URL", "https://rsgd15-foundry.openai.azure.com/openai/v1/")
MODEL          = "gpt-5"
ARENA_URL      = "https://agent-coliseum.onrender.com"
NGROK_TOKEN    = os.environ.get("NGROK_TOKEN", "YOUR_NGROK_TOKEN_HERE")

# Azure Foundry uses the standard OpenAI client with a custom base_url
client = OpenAI(base_url=AZURE_BASE_URL, api_key=AZURE_API_KEY)
print(f"Client ready: {client.base_url}")

# ── LangChain-style chain (plain openai, no transformers dependency) ───────
# Mimics LangChain LCEL: prompt | llm | StrOutputParser
# Avoids langchain_openai import which conflicts with Azure ML transformers

SYSTEM_PROMPT = """You are a competitive Latin America knowledge agent.
You think carefully before asking or answering.
Use this 6-step reasoning structure:

# SITUATION  [Chain-of-Thought — Wei et al. 2022, NeurIPS]
SITUATION: <assess topic, role, turn, score gap>

# OPPONENT  [Theory of Mind — Langner et al. 2023]
OPPONENT: <model their knowledge gaps and tendencies>

# GOAL  [ReAct — Yao et al. 2022, ICLR 2023]
GOAL: <one concrete objective for this turn>

# DRAFT  [Scratchpad — Nye et al. 2021]
DRAFT: <first attempt>

# CRITIQUE  [Self-Refine — Madaan et al. 2023, NeurIPS]
CRITIQUE: <evaluate draft quality>

# FINAL  [Reflexion — Shinn et al. 2023]
FINAL: <final question (1 sentence) or answer (1-2 sentences max)>"""

def think_chain_invoke(input_text: str) -> str:
    """Mimics LangChain LCEL: prompt | llm | StrOutputParser."""
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": input_text},
        ],
        max_tokens=400,
    )
    return resp.choices[0].message.content.strip()

# ── Agent ─────────────────────────────────────────────────────────────────

class LangChainLatAmAgent(Agent):
    """
    LangChain-style agent using plain openai client.
    Maintains per-match conversation memory as a plain list.
    Demonstrates structured reasoning without external dependencies.
    """

    name        = "LangChain Puma"
    avatar      = "🐆"
    description = "Agente con memoria de conversacion y razonamiento estructurado"

    def __init__(self):
        self._match_memory   = {}
        self._opponent_notes = {}

    def on_arena_start(self, ctx: WorldContext) -> None:
        self._match_memory   = {}
        self._opponent_notes = {}
        print(f"[{self.name}] Arena started with {len(ctx.agents)} agents.")

    def on_match_start(self, ctx: MatchContext) -> None:
        self._match_memory[ctx.match_id] = []
        print(f"[{self.name}] Starting match {ctx.match_id} vs {ctx.opponent_name}")

    def on_match_end(self, ctx: MatchContext, result: MatchResult) -> None:
        won = result.winner_id == ctx.my_agent_id
        self._opponent_notes[ctx.opponent_agent_id] = {
            "name":   ctx.opponent_name,
            "result": "won" if won else "lost",
            "topic":  ctx.topic,
        }
        print(f"[{self.name}] Match over: {'WON' if won else 'LOST'}")

    def move(self, ctx: WorldContext) -> Position:
        dx, dy = random.choice([(0,1),(0,-1),(1,0),(-1,0),(0,0)])
        return Position(x=max(0, min(ctx.map_width-1,  ctx.my_position.x+dx)),
                        y=max(0, min(ctx.map_height-1, ctx.my_position.y+dy)))

    def think(self, ctx: MatchContext) -> str:
        history = ""
        for t in ctx.history[-3:]:
            history += f"\n  Turn {t['turn_number']}: Q={t['question'][:50]} A={t['answer'][:50]} Score={t['score']}"

        opp = self._opponent_notes.get(ctx.opponent_agent_id, {})
        opp_text = (f"Previously: {opp.get('result','unknown')} on {opp.get('topic','?')}"
                    if opp else "First encounter.")

        mem    = self._match_memory.get(ctx.match_id, [])
        mem_txt = "".join(f"\n  Turn {e['turn']} ({e['role']}): {e['summary']}"
                          for e in mem[-2:])

        input_text = f"""MATCH:
Topic: {ctx.topic} | Role: {ctx.role} | Turn {ctx.turn}/{ctx.total_turns}
My score: {sum(ctx.my_scores)} pts | Opponent: {sum(ctx.opponent_scores)} pts
Opponent: {ctx.opponent_name}
{f"Question: {ctx.current_question}" if ctx.role == "answerer" else ""}

OPPONENT: {opp_text}
HISTORY:{history if history else " (first turn)"}
MY MEMORY:{mem_txt if mem_txt else " (empty)"}

Think through SITUATION / OPPONENT / GOAL / DRAFT / CRITIQUE / FINAL."""

        result = think_chain_invoke(input_text)

        mem.append({"turn": ctx.turn, "role": ctx.role, "summary": result[:150]})
        self._match_memory[ctx.match_id] = mem
        return result

    def ask(self, ctx: MatchContext) -> str:
        return self._extract_final(self.think(ctx))

    def answer(self, ctx: MatchContext) -> str:
        return self._extract_final(self.think(ctx))

    def _extract_final(self, text: str) -> str:
        """
        Extract the FINAL answer from the reasoning trace.
        Robust to GPT formatting: "FINAL: ...", "**FINAL:**", FINAL on own line.
        Falls back to last non-empty non-comment line.
        """
        lines = text.split("\n")
        for i, line in enumerate(lines):
            clean = line.replace("**", "").strip()
            if clean.upper().startswith("FINAL:"):
                rest = clean[6:].strip()
                if rest:
                    return rest
                for j in range(i+1, len(lines)):
                    if lines[j].strip():
                        return lines[j].strip()
            elif clean.upper() == "FINAL":
                for j in range(i+1, len(lines)):
                    if lines[j].strip():
                        return lines[j].strip()
        for line in reversed(lines):
            if line.strip() and not line.strip().startswith("#"):
                return line.strip()
        return text.strip()

# ── Keepalive ─────────────────────────────────────────────────────────────
import threading, requests as _req, time as _time

def _keepalive(arena_url):
    """Ping arena every 60s to keep ngrok tunnel and Colab session alive."""
    while True:
        _time.sleep(60)
        try:
            _req.get(f"{arena_url}/health", timeout=5)
        except:
            pass

threading.Thread(target=_keepalive, args=(ARENA_URL,), daemon=True).start()
print("Keepalive thread started")

# ── Run ───────────────────────────────────────────────────────────────────
agent = LangChainLatAmAgent()

In [ ]:
# ── DEBUG CELL — run this to inspect raw GPT-5 output ────────────────────
# This helps verify that _extract_final works correctly.
# Run this cell after the agent is defined, before serve_and_register.

from agent_base import MatchContext

test_ctx = MatchContext(
    match_id="test", topic="Latin American geography",
    turn=1, total_turns=3, role="asker",
    history=[], my_agent_id="me", opponent_agent_id="opp",
    opponent_name="Test Opponent", my_scores=[], opponent_scores=[],
)

print("=== RAW think() OUTPUT ===")
raw = agent.think(test_ctx)
print(raw)
print()
print("=== EXTRACTED FINAL ===")
print(repr(agent._extract_final(raw)))
print()

test_ctx_answerer = MatchContext(
    match_id="test", topic="Latin American geography",
    turn=1, total_turns=3, role="answerer",
    history=[], my_agent_id="me", opponent_agent_id="opp",
    opponent_name="Test Opponent", my_scores=[], opponent_scores=[],
    current_question="What is the longest river in South America?",
)

print("=== RAW think() OUTPUT (answerer) ===")
raw2 = agent.think(test_ctx_answerer)
print(raw2)
print()
print("=== EXTRACTED FINAL ===")
print(repr(agent._extract_final(raw2)))

In [ ]:
# ── Run cell — starts the agent server ───────────────────────────────────
from pyngrok import ngrok
ngrok.kill()  # kill any existing tunnels before starting

serve_and_register(
    agent       = agent,
    arena_url   = ARENA_URL,
    port        = 5000,
    ngrok_token = NGROK_TOKEN,
)
# This cell blocks. Agent is now live and registered.
# Wait for the organizer to accept you in the admin panel.